# Bielik 1.5B → model skojarzeń LanguaLinker (ADR-23)

Douczanie LoRA modelu `speakleash/Bielik-1.5B-v3.0-Instruct` na skojarzeniach z talii, ewaluacja, scalenie, konwersja do GGUF (`Q4_K_M`) i manifest do GitHub Release.

**Środowisko:** Google Colab z GPU (T4 wystarczy, ok. 30–60 min): *Runtime → Change runtime type → T4 GPU*.

**Wejście:** `train.jsonl` i `eval.jsonl` z eksportera:
```
cargo run -p langualinker-application --bin llm-dataset -- content/decks target/llm-dataset
```
Pliki zawierają treść talii, która w repozytorium jest zaszyfrowana — nie publikuj ich i usuń z Colaba po treningu.

**Wyjście:** `bielik-1.5b-associations-v1.Q4_K_M.gguf` i `model-manifest.json` (wydanie na GitHubie — sekcja 9).

## 1. Konfiguracja

In [ ]:
MODEL_ID = 'speakleash/Bielik-1.5B-v3.0-Instruct'   # model bazowy (Apache 2.0; na HF trzeba zaakceptować warunki)
RUN_NAME = 'bielik-1.5b-associations-v1'           # id modelu w aplikacji i nazwa pliku
QUANT = 'Q4_K_M'                                   # kwantyzacja GGUF dla telefonu (~1 GB)
RELEASE_TAG = 'model-bielik-associations-v1'       # tag wydania na GitHubie
REPO = 'borjarus/langua-linker'

EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_RANK = 16
MAX_LENGTH = 1024        # tokeny na przykład (system + karta + odpowiedź)
BATCH_SIZE = 8
GRAD_ACCUM = 2
EVAL_SAMPLES = 120       # ile przykładów z eval.jsonl generować w ewaluacji

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q -U "transformers>=4.46" "peft>=0.13" "accelerate>=1.0" sentencepiece huggingface_hub

## 2. Dostęp do modelu bazowego

Model na Hugging Face wymaga zaakceptowania warunków na stronie modelu i tokenu (*Settings → Access Tokens*, uprawnienie *read*). W Colabie dodaj go jako sekret `HF_TOKEN` (ikona klucza po lewej).

In [ ]:
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass
login(token=os.environ.get('HF_TOKEN'))

## 3. Dane

In [ ]:
import json
from collections import Counter

if not (os.path.exists('train.jsonl') and os.path.exists('eval.jsonl')):
    from google.colab import files
    print('Wybierz train.jsonl i eval.jsonl z target/llm-dataset')
    files.upload()

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows, eval_rows = read_jsonl('train.jsonl'), read_jsonl('eval.jsonl')
print(len(train_rows), 'train /', len(eval_rows), 'eval')
print(Counter((r['meta']['target'], r['meta']['method']) for r in train_rows))

## 4. Tokenizacja

Prompt to szablon czatu modelu (ChatML: `<|im_start|>…<|im_end|>`). Strata liczona jest **tylko na odpowiedzi** — prompt systemowy i dane karty są maskowane (`-100`).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
END = '<|im_end|>'

def encode(row):
    messages = row['messages']
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full = prompt + messages[-1]['content'] + END
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    ids = tokenizer(full, add_special_tokens=False)['input_ids'][:MAX_LENGTH]
    labels = ([-100] * len(prompt_ids) + ids[len(prompt_ids):])[:len(ids)]
    return {'input_ids': ids, 'labels': labels}

train_enc = [encode(r) for r in train_rows]
eval_enc = [encode(r) for r in eval_rows]
lengths = sorted(len(e['input_ids']) for e in train_enc)
print('długość: mediana', lengths[len(lengths) // 2], 'max', lengths[-1])
assert all(any(l != -100 for l in e['labels']) for e in train_enc), 'odpowiedź ucięta przez MAX_LENGTH'
print(tokenizer.decode(train_enc[0]['input_ids'])[-300:])

## 5. Trening LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

bf16 = torch.cuda.is_bf16_supported()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if bf16 else torch.float16, device_map='auto')
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM'))
model.print_trainable_parameters()

def collate(batch):
    width = max(len(b['input_ids']) for b in batch)
    pad = lambda seq, value: seq + [value] * (width - len(seq))
    return {
        'input_ids': torch.tensor([pad(b['input_ids'], tokenizer.pad_token_id) for b in batch]),
        'labels': torch.tensor([pad(b['labels'], -100) for b in batch]),
        'attention_mask': torch.tensor([pad([1] * len(b['input_ids']), 0) for b in batch]),
    }

args = TrainingArguments(
    output_dir='lora', num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM, lr_scheduler_type='cosine', warmup_ratio=0.03,
    logging_steps=20, eval_strategy='epoch', save_strategy='no',
    bf16=bf16, fp16=not bf16, report_to='none', remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_enc, eval_dataset=eval_enc, data_collator=collate)
trainer.train()
print(trainer.evaluate())

## 6. Ewaluacja na odłożonych kartach

Te same kontrole co w aplikacji (ADR-18): odpowiedź to obiekt `{"association": "..."}`, niepusta, do 400 znaków, bez adresów URL. Do tego: czy skojarzenie wymienia słowo z karty. Przejrzyj też ręcznie przykłady w następnej komórce — liczby nie powiedzą, czy skojarzenie jest *dobre*.

In [ ]:
import re, random
model.eval()
random.seed(7)
sample = random.sample(eval_rows, min(EVAL_SAMPLES, len(eval_rows)))
end_id = tokenizer.convert_tokens_to_ids(END)

def generate(messages, temperature=0.7):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=temperature, top_p=0.9,
                             eos_token_id=end_id, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def parse(answer):
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', answer.strip())
    try:
        value = json.loads(text)['association']
        return value if isinstance(value, str) else None
    except Exception:
        return None

results = []
for row in sample:
    card = json.loads(row['messages'][1]['content'].split('\n', 1)[1])
    raw = generate(row['messages'][:2])
    results.append({
        'target': row['meta']['target'], 'method': row['meta']['method'],
        'text': card['text'], 'meaning': card['meaning'], 'raw': raw, 'association': parse(raw),
        'reference': json.loads(row['messages'][2]['content'])['association'],
    })

def rate(predicate):
    return f"{sum(1 for r in results if predicate(r)) / len(results):.0%}"

first_word = lambda r: re.sub(r'[*.,!?¿¡]', '', r['text']).split()[0].lower()
print('poprawny JSON       ', rate(lambda r: r['association'] is not None))
print('niepuste, ≤400 znaków', rate(lambda r: bool(r['association']) and len(r['association']) <= 400))
print('bez URL             ', rate(lambda r: r['association'] is not None and 'http' not in r['association']))
print('wymienia słowo      ', rate(lambda r: bool(r['association']) and first_word(r) in r['association'].lower()))

In [ ]:
for r in results[:25]:
    print(f"[{r['target']}/{r['method']}] {r['text']} — {r['meaning']}")
    print('  model:  ', r['association'] if r['association'] is not None else '(niepoprawne) ' + r['raw'][:200])
    print('  wzorzec:', r['reference'])
    print()

## 7. Scalenie i konwersja do GGUF

Scala adapter LoRA z modelem bazowym, konwertuje skryptem llama.cpp do GGUF (f16) i kwantyzuje do `Q4_K_M`. Na końcu krótki test pliku tym samym silnikiem, którego użyje aplikacja (llama.cpp).

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained('merged', safe_serialization=True)
tokenizer.save_pretrained('merged')
del merged, model, trainer
torch.cuda.empty_cache()

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build llama.cpp/build --target llama-quantize llama-cli -j 8 > /dev/null
!python llama.cpp/convert_hf_to_gguf.py merged --outtype f16 --outfile {RUN_NAME}.f16.gguf
!llama.cpp/build/bin/llama-quantize {RUN_NAME}.f16.gguf {RUN_NAME}.{QUANT}.gguf {QUANT}
!ls -la *.gguf

In [ ]:
prompt = tokenizer.apply_chat_template(eval_rows[0]['messages'][:2], tokenize=False, add_generation_prompt=True)
open('prompt.txt', 'w', encoding='utf-8').write(prompt)
!llama.cpp/build/bin/llama-cli -m {RUN_NAME}.{QUANT}.gguf -f prompt.txt -n 200 --temp 0.7 -no-cnv -r '<|im_end|>' 2>/dev/null | tail -5

## 8. Manifest dla aplikacji

In [ ]:
import hashlib, datetime
gguf = f'{RUN_NAME}.{QUANT}.gguf'
digest = hashlib.sha256()
with open(gguf, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        digest.update(chunk)
manifest = {
    'id': RUN_NAME,
    'name': 'Bielik 1.5B (skojarzenia)',
    'file': gguf,
    'url': f'https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{gguf}',
    'sha256': digest.hexdigest(),
    'bytes': os.path.getsize(gguf),
    'base': MODEL_ID,
    'license': 'Apache-2.0',
    'chatTemplate': 'chatml',
    'promptVersions': sorted({r['meta']['promptVersion'] for r in train_rows}),
    'contextTokens': 2048,
    'trainedAt': datetime.date.today().isoformat(),
    'trainExamples': len(train_rows),
}
json.dump(manifest, open('model-manifest.json', 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 9. Wydanie na GitHubie

Pobierz `*.Q4_K_M.gguf` i `model-manifest.json` (panel plików po lewej → *Download*, albo kopia na Google Drive w ostatniej komórce) i na komputerze z zalogowanym `gh`:
```
gh release create model-bielik-associations-v1 bielik-1.5b-associations-v1.Q4_K_M.gguf model-manifest.json \
  --repo borjarus/langua-linker --prerelease --title "On-device association model v1" \
  --notes "Bielik 1.5B v3.0 Instruct + LoRA; see docs/adr/ADR-23-on-device-associations.md"
```
Potem dopisz manifest do tablicy `models` w `content/models/catalog.json` — aplikacja oferuje tylko modele z katalogu i sprawdza SHA-256 pobranego pliku.

**Uwaga:** repozytorium jest publiczne, więc plik modelu też będzie publiczny. Przeczytaj sekcję *Publikacja* w ADR-23 przed wydaniem.

In [ ]:
# Opcjonalnie: kopia na Google Drive (pliki ~1 GB pobierają się z Colaba wolno).
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/langualinker && cp {RUN_NAME}.{QUANT}.gguf model-manifest.json /content/drive/MyDrive/langualinker/